# 092 — Generación y edición de video

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**El problema temporal**: un video es un tensor F×H×W×3. Generar frames independientes
produce parpadeo — identidad, textura e iluminación cambian entre frames. El modelo
necesita dependencias temporales explícitas.

**Difusión de video latente**: un VAE comprime cada frame (H/8 × W/8 típico) y la
difusión opera sobre el tensor latente completo. Stable Video Diffusion *infla* una
U-Net de imagen preentrenada insertando capas temporales y la ajusta con video curado.
Make-A-Video aprende apariencia de pares texto-imagen y movimiento de video sin texto.

**Atención espacio-temporal**: con F frames de h×w latentes hay n = F·h·w tokens.
Atención completa: O(n²). Factorizada: espacial F·(h·w)² + temporal h·w·F² — mucho más
barata, pero un token solo ve otro frame Y otra posición en dos saltos.

**Edición**: *video2video* parte del video ruidificado a un nivel intermedio (el nivel
de ruido controla fidelidad ↔ libertad); la *propagación de ediciones* edita un frame
clave y lo propaga por correspondencias temporales (falla con oclusiones y cortes de
plano).


## 🧮 Ejemplo de referencia

16 frames de 32×32 latentes → n = 16 × 1 024 = 16 384 tokens. Atención completa:
16 384² ≈ **2.68 × 10⁸** pares. Factorizada: 16·(1024²) + 1024·(16²) =
16 777 216 + 262 144 ≈ **1.70 × 10⁷**. Razón ≈ **15.8×** — y el ahorro crece con la
longitud del clip. Reprodúcelo a mano antes de ejecutar el laboratorio.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("generation", seed=92)
show(result)


## Reflexión

1. En la atención factorizada, un token del frame 3 en la posición (0,0) y otro del
   frame 9 en la posición (31,31) nunca se atienden directamente: ¿por qué camino les
   llega la información y qué tipo de movimiento podría degradarse por ello?
2. Make-A-Video aprende apariencia de pares texto-imagen y movimiento de video SIN
   texto: ¿qué suposición sobre la factorización apariencia/movimiento hace posible
   ese truco y cuándo fallaría?
3. En video2video, ¿por qué ruidificar poco preserva el video original y ruidificar
   mucho lo destruye, y cómo elegirías el nivel para "cambiar el estilo sin cambiar
   el movimiento"?
